# GPU worker — discover Dataset, re-hash, FP16 batch smoke-v2

Do not start GPU work until `artifact_gate.py <dir>` exits 0.
`/tmp` is same-session only. Prefer `/kaggle/input/council-qwen05`.

In [ ]:
import json, os, shutil, signal, subprocess, sys, time, hashlib
from pathlib import Path

WORKING = Path("/kaggle/working")
WORKING.mkdir(parents=True, exist_ok=True)
for name in ("worker_entry.py", "smoke_scorer.py", "artifact_gate.py"):
    src = Path(name)
    if src.exists():
        shutil.copy(src, WORKING / name)
sys.path.insert(0, str(WORKING))
import artifact_gate, smoke_scorer

DATASET_ROOT = Path("/kaggle/input/council-qwen05")
if DATASET_ROOT.exists():
    MODEL_DIR = artifact_gate.find_complete_model_dir(str(DATASET_ROOT))
else:
    alt = Path("/tmp/modelscope_cache/Qwen/Qwen2.5-0.5B-Instruct")
    if not alt.is_dir():
        raise FileNotFoundError("attach council-qwen05 or download in THIS session")
    MODEL_DIR = artifact_gate.find_complete_model_dir(str(alt))
print("Resolved model directory:", MODEL_DIR)

gate = subprocess.run([sys.executable, str(WORKING/"artifact_gate.py"), MODEL_DIR])
if gate.returncode != 0:
    raise SystemExit("artifact_gate failed; do not start worker")

BENCH = Path("datasets/smoke-v2.jsonl")
if not BENCH.exists():
    BENCH = WORKING / "datasets" / "smoke-v2.jsonl"
items = smoke_scorer.load_smoke(str(BENCH))
smoke_scorer.validate_benchmark(items)
bench_hash = hashlib.sha256(BENCH.read_bytes()).hexdigest()
print("benchmark", len(items), bench_hash)

request = {
    "model": MODEL_DIR,
    "benchmark_path": str(BENCH),
    "output_predictions_path": str(WORKING / "predictions.jsonl"),
    "max_input_tokens": 256,
    "max_new_tokens": 64,
    "local_files_only": True,
    "dtype": "float16",
    "quantization": "none",
}
req = WORKING / "worker_request.json"
res = WORKING / "worker_result.json"
log = WORKING / "worker.log"
req.write_text(json.dumps(request, indent=2))
if res.exists(): res.unlink()

timed_out = False
with log.open("wb") as fh:
    proc = subprocess.Popen(
        [sys.executable, str(WORKING/"worker_entry.py"), str(req), str(res)],
        stdout=fh, stderr=subprocess.STDOUT, start_new_session=True,
    )
    try:
        proc.wait(timeout=1800)
    except subprocess.TimeoutExpired:
        timed_out = True
        os.killpg(proc.pid, signal.SIGTERM); time.sleep(5)
        if proc.poll() is None:
            os.killpg(proc.pid, signal.SIGKILL)
        proc.wait()
        (WORKING / "failure.marker").write_text("timeout\n")
        raise RuntimeError("worker timed out")

if not res.exists():
    tail = log.read_text(encoding="utf-8", errors="replace")[-4000:] if log.exists() else ""
    (WORKING / "failure.marker").write_text(f"no result rc={proc.returncode}\n")
    raise RuntimeError(f"worker_result.json missing rc={proc.returncode}\n{tail}")

result = json.loads(res.read_text())
expected = len(items)
ev = result.get("evaluation") or {}
run_success = (
    proc.returncode == 0
    and result.get("ok") is True
    and result.get("mode") == "benchmark"
    and result.get("benchmark_items_attempted") == expected
    and result.get("predictions_written") == expected
    and ev.get("total") == expected
)
if ev:
    (WORKING / "evaluation.json").write_text(json.dumps(ev, indent=2))
report = {
    "schema_version": 2,
    "run_id": "qwen05-smoke-v2-fp16",
    "status": "success" if run_success else "failure",
    "model": {
        "seat_id": "qwen25-05b",
        "revision": "master",
        "official_revision_pinned": False,
        "weight_sha256": "fdf756fa7fcbe7404d5c60e26bff1a0c8b8aa1f72ced49e7dd0210fe288fb7fe",
        "dtype": "float16",
        "quantization": "none",
    },
    "benchmark": {
        "name": "smoke-v2",
        "records": expected,
        "sha256": bench_hash,
    },
    "environment": result.get("versions"),
    "process": {"return_code": proc.returncode, "timed_out": timed_out},
    "outputs": {
        "predictions_written": result.get("predictions_written", 0),
        "prediction_failures": result.get("prediction_failures", 0),
        "evaluation_written": ev.get("total") == expected,
    },
    "evaluation": ev,
    "memory": {
        "peak_allocated_bytes": result.get("peak_allocated_bytes"),
        "peak_reserved_bytes": result.get("peak_reserved_bytes"),
    },
    "timing": {
        "load_seconds": result.get("load_seconds"),
        "inference_seconds": result.get("inference_seconds"),
        "wall_seconds": result.get("wall_seconds"),
    },
    "promotion": {"eligible": False, "decision": "baseline-only"},
}
(WORKING / "run_report.json").write_text(json.dumps(report, indent=2))
marker = WORKING / ("completion.marker" if run_success else "failure.marker")
marker.write_text(report["status"] + "\n")
print(json.dumps({"run_success": run_success, "rc": proc.returncode,
                  "score": ev.get("score"), "status": report["status"]}, indent=2))
if not run_success:
    raise RuntimeError(result.get("error") or "run_success false")